# A 25 m terrain for GeoModeller

Default example: **Cervin / Matterhorn, 6 x 6 km**. This notebook downloads swissALTI3D, averages it to 25 m, and exports an ASCII terrain plus a matching shaded relief PNG.

**Edit only the rectangle below, then run all cells from top to bottom.** Use the **Python (swiss-dem)** kernel. On NOTO, calculations and files stay on the server until you download the final ZIP. See the README for setup and GeoModeller import.


## 1. EDIT HERE — study area

Keep these values for the Cervin test. For another area, enter **`(xmin, ymin, xmax, ymax)` in LV95 metres**, with each side between 100 m and 10 km. The output cell size stays at 25 m. These are coordinates, not width/height values.


In [ ]:
# ONLY CHANGE THIS RECTANGLE — default: Cervin / Matterhorn.
AOI_LV95 = (2_614_000, 1_089_000, 2_620_000, 1_095_000)


## 2. Setup and validate — run unchanged

Locate the project and validate the rectangle before downloading. Implementation helpers live in `dem_tools.py`; keep that file alongside the README when copying this project.


In [ ]:
from pathlib import Path
import sys

# Works from the project root or notebooks/, including a GitHub ZIP folder.
PROJECT_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
                     if (p / "dem_tools.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook inside the complete get_dem project folder.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dem_tools import validate_aoi, find_tiles, download_tiles, create_grid, export_outputs, preview

bounds = validate_aoi(AOI_LV95)
CACHE = PROJECT_ROOT / "data" / "swissalti3d_tiles"
OUTPUT = PROJECT_ROOT / "output"
print(f"Area: {(bounds[2]-bounds[0])/1000:g} x {(bounds[3]-bounds[1])/1000:g} km")
print(f"Python: {sys.executable}")


## 3. Download the source — run unchanged

Internet access is required for the swisstopo catalog and uncached tiles. Only 2 m LV95/LN02 tiles are selected, using the latest available edition of each tile. Downloads are checked before being reused. Source TIFFs stay in the cache; GeoModeller does not need them.


In [ ]:
tiles = find_tiles(bounds)
print(f"Source tiles: {len(tiles)}")
tile_paths = download_tiles(tiles, CACHE)


## 4. Build the terrain and hillshade — run unchanged

Average the 2 m data once onto 25 m cells. The output rectangle expands to whole cells if necessary. Missing terrain is kept as NoData, not filled or interpolated across borders. The PNG is calculated from the terrain without another download.

This step replaces the named files in `output`; download or rename a previous result before changing the area.


In [ ]:
dem, transform = create_grid(tile_paths, bounds)
info, rgba, archive = export_outputs(dem, transform, OUTPUT, tiles)
print(f"Grid: {dem.shape[1]} columns x {dem.shape[0]} rows at 25 m")
print(f"Valid terrain coverage: {info['coverage_percent']:.1f}%")
if info['coverage_percent'] < 100:
    print("Some cells have no data (for example beyond Swiss coverage). They remain empty.")
print(f"Download this file after checking the preview: {archive}")


## 5. Project limits for GeoModeller — copy all six

Select **CH1903+ / LV95 (EPSG:2056)** and metre units in GeoModeller, then enter these absolute limits. Do not keep a project extent of 0–1000. ZMin/ZMax surround the terrain with at least 100 m margin; lower ZMin if your geological model should extend deeper.

The same values and the image registration coordinates are saved in `project_info.txt` inside the ZIP. For the default Cervin example, expect a **240 x 240** grid. The 25 m resolution describes each cell, not the total area.


In [ ]:
for axis in "XYZ":
    print(f"{axis}Min = {info[axis+'Min']:.2f}    {axis}Max = {info[axis+'Max']:.2f}")
print(f"Terrain elevations (LN02): {info['elevation_min']:.2f} to {info['elevation_max']:.2f} m")


## 6. Check and download

The hillshade is a map viewed from above, not a perspective view of the mountain. Lighting is from the northwest, at 45 degrees; gaps are transparent. Neither display changes the exported heights.

After checking the preview, download **`output/geomodeller_dem.zip`** from NOTO's file browser and extract it on the GeoModeller computer. Import the `.asc` as **Digital Terrain Model / Arc Grid**. Add the `.png` through **Image Manager** on `SurfaceTopography` if wanted. Keep the companion files together. An image alone does not create a 3D surface.


In [ ]:
preview(dem, rgba, transform)
